In [2]:
import torch
import torch.nn as nn
import torch.optim as optim


In [5]:


# creating the tensors

X = torch.linspace(-5, 5, 100).view(-1, 5)
Y = torch.rand(X.shape[0],1)

print(X.shape, Y.shape)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


X_train, X_test, Y_train, Y_test = train_test_split(X.numpy(),Y.numpy(),test_size=0.2, random_state=42) 


# Preprocessing the data
preprocessor = ColumnTransformer(
    transformers=[
        ('imputer', SimpleImputer(strategy='mean'), slice(0, X.shape[1])),
        ('scaler', StandardScaler(), slice(0, X.shape[1]))
        
    ]
)

X_train = torch.tensor(preprocessor.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(preprocessor.transform(X_test), dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32)
Y_test  = torch.tensor(Y_test, dtype=torch.float32)

#Creatind into dataset and dataloader
dataset = torch.utils.data.TensorDataset(X_train,Y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=10, shuffle=True)


# model initialization
model = nn.Linear(X_train.shape[1], 1)

epsilon = 0.1
c = 1.0

optimizer = optim.Adam(model.parameters(), lr=0.01)


# Model training  

epochs = 10000
for epoch in range(epochs):
    total_loss = 0.0
    for inputs, targets in dataloader:
        #forward pass
        Y_pred = model(inputs)
        
        abs_loss = torch.abs(targets - Y_pred)
        
        epsilon_loss = torch.where(abs_loss < epsilon, torch.zeros_like(abs_loss), abs_loss - epsilon)
        
        L2_reg = torch.sum(model.weight ** 2) * 0.5
        
        loss = torch.mean(epsilon_loss) + c * L2_reg
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        if epoch % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}')


torch.Size([20, 5]) torch.Size([20, 1])
Epoch [1/10000], Loss: 0.5319
Epoch [1/10000], Loss: 0.7737
Epoch [101/10000], Loss: 0.1005
Epoch [101/10000], Loss: 0.1640
Epoch [201/10000], Loss: 0.0804
Epoch [201/10000], Loss: 0.1782
Epoch [301/10000], Loss: 0.0930
Epoch [301/10000], Loss: 0.1630
Epoch [401/10000], Loss: 0.0784
Epoch [401/10000], Loss: 0.1789
Epoch [501/10000], Loss: 0.0799
Epoch [501/10000], Loss: 0.1750
Epoch [601/10000], Loss: 0.1288
Epoch [601/10000], Loss: 0.1513
Epoch [701/10000], Loss: 0.0773
Epoch [701/10000], Loss: 0.1778
Epoch [801/10000], Loss: 0.0769
Epoch [801/10000], Loss: 0.1789
Epoch [901/10000], Loss: 0.0961
Epoch [901/10000], Loss: 0.1678
Epoch [1001/10000], Loss: 0.0777
Epoch [1001/10000], Loss: 0.1731
Epoch [1101/10000], Loss: 0.0674
Epoch [1101/10000], Loss: 0.1834
Epoch [1201/10000], Loss: 0.1083
Epoch [1201/10000], Loss: 0.1641
Epoch [1301/10000], Loss: 0.1034
Epoch [1301/10000], Loss: 0.1624
Epoch [1401/10000], Loss: 0.0770
Epoch [1401/10000], Loss: 0